In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler , OneHotEncoder
from sklearn.decomposition import PCA
from sklearn.model_selection import cross_validate , train_test_split , KFold , RandomizedSearchCV , StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from imblearn.pipeline import Pipeline
from sklearn.base import BaseEstimator , TransformerMixin
 
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC , LinearSVC
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE
import scipy
from dataclasses import dataclass
from MlUtlitys import PlotManager

ModuleNotFoundError: No module named 'MlUtlitys'

In [ ]:
linPath = r"/run/media/drdrakken/Elements/Sonstiges/Programmieren/Machine Learning/csvs/Competition/GiveMeSomeCredit/cs-training.csv"
winPath = r"I:\Sonstiges\Programmieren\Machine Learning\csvs\Competition\GiveMeSomeCredit\cs-training.csv"
df = pd.read_csv(linPath)

In [ ]:
@dataclass
class Config():
    target = str = "SeriousDlqin2yrs"
    seed = 1234
    test_size = 0.2
    cross_iterations:int = 5
    verbose:int = 0
    grid_test_amount:int = 20
    grid_scoring :str = "roc_auc"
config = Config()

In [ ]:
class DataClass():

    def __init__(self , Data ):
        self.data = Data.copy()

        self.numericalData = self.data.select_dtypes(include = "number").columns
        self.categoricalData = self.data.select_dtypes(exclude = "number").columns

        self.x = self.data.drop([Config.target] , axis = 1)
        self.y = self.data[config.target]
        
        self.Stats()
        self.DataInfo()

    def Stats(self):
        print(f"XShape:{self.x.shape}")
        print(f"YShape:{self.y.shape}")
        print(f"ClassCount:{self.y.value_counts()}")

    def DataInfo(self):
        print(f"Infos:{self.data.info()}")
        print(f"Description:{self.data.describe()}")
        print(f"First Cols:{self.data[:20]}")
        
data = DataClass(Data = df)

XShape:(150000, 11)
YShape:(150000,)
ClassCount:SeriousDlqin2yrs
0    139974
1     10026
Name: count, dtype: int64
<class 'pandas.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 12 columns):
 #   Column                                Non-Null Count   Dtype  
---  ------                                --------------   -----  
 0   Unnamed: 0                            150000 non-null  int64  
 1   SeriousDlqin2yrs                      150000 non-null  int64  
 2   RevolvingUtilizationOfUnsecuredLines  150000 non-null  float64
 3   age                                   150000 non-null  int64  
 4   NumberOfTime30-59DaysPastDueNotWorse  150000 non-null  int64  
 5   DebtRatio                             150000 non-null  float64
 6   MonthlyIncome                         120269 non-null  float64
 7   NumberOfOpenCreditLinesAndLoans       150000 non-null  int64  
 8   NumberOfTimes90DaysLate               150000 non-null  int64  
 9   NumberRealEstateLoansOrLines    

In [ ]:
class DataPreprcocess():

    def fit(self , X , y = None):
        numerical_cols = X.select_dtypes(include = np.number).columns
        categorical_cols = X.select_dtypes(exclude = np.number).columns

        self.preprocess = ColumnTransformer([
            ("numerical_data_process" , Pipeline([
                ("imputer" , SimpleImputer(strategy = "mean")),
                ("scale" , MinMaxScaler()),
            ]),numerical_cols),

            ("categorical_data_process" , Pipeline([
                ("imputer" , SimpleImputer(strategy = "most_frequent")),
                ("encoder" , OneHotEncoder(handle_unknown = "ignore" , sparse_output=False)),
            ]),categorical_cols),
        ])
        self.preprocess.fit(X)
        return self
    
    def transform(self , X , y = None):
        return self.preprocess.transform(X)

In [ ]:
class Visualize():

    def __init__(self , Data):
        self.data = Data.copy()

    def IQR(self , TargetCol):
        q1 =  self.data[TargetCol].quantile(0.25)
        q3 = self.data[TargetCol].quantile(0.75)
        iqr = q3 - q1
        return q1 , q3 , iqr
    
    def Skew(self , SkewCol):
        dataSkew = self.data.skew()
        print(f"Skewness of Col:->{SkewCol}:->{dataSkew}")
    
    def PlotData(self):
        for vis in self.data.numericalData:
            fig , axes = plt.subplots(3 , 1,  figsize = (10 , 10 ) , dpi = 200)
            q1,  q3 , _ = self.IQR(TargetCol = vis)
            mean = self.data[mean]
            self.Skew(SkewCol = vis)

            sns.histplot(data = self.data , x = self.data.x , y = self.data.y , ax = axes[0])
            axes[0].axvline(q1 , color = "green")
            axes[0].axvline(q3 , color = "red")
            axes[0].axvline(mean , color = "yellow")
            axes[0].set_title(f"Histplot for :{vis}")

            sns.histplot(data = self.data , x = self.data.x , y = self.data.y , ax = axes[0])
            axes[1].axvline(q1 , color = "green")
            axes[1].axvline(q3 , color = "red")
            axes[1].axvline(mean , color = "yellow")
            axes[1].set_title(f"Histplot for :{vis}")

            sns.histplot(data = self.data , x = self.data.x , y = self.data.y , ax = axes[0])
            axes[2].axvline(q1 , color = "green")
            axes[2].axvline(q3 , color = "red")
            axes[2].axvline(mean , color = "yellow")
            axes[2].set_title(f"Histplot for :{vis}")

            plt.tight_layout()
            plt.show()

In [ ]:
class DataTransform(BaseEstimator , TransformerMixin):

    def fit(self, X , y = None):
        return self
    
    def transform(self, X , y = None):
        x = X.copy()
        x = self.NewFeatures(X)
        return x
    
    def NewFeatures(self , X ):
        x = X.copy()
        
        return x

In [ ]:
def model_varianz():
    return {

        "LogisticRegression":LogisticRegression(random_state = config.seed , max_iter = 1000),
        "SVC":SVC(random_state = config.seed , kernel = "linear" , probability = True ),
        "RandomForestClassifier":RandomForestClassifier(random_state = config.seed),
        "DecisionTreeClassifier":DecisionTreeClassifier(random_state = config.seed),
        
    }

In [ ]:
def custom_grid_search(estimator , parameters , X_train , y_train):
    grid = RandomizedSearchCV(estimator= estimator , 
                              return_train_score= True,
                              param_distributions = parameters,
                              random_state = config.seed, 
                              cv = config.cross_iterations,
                              n_iter = config.grid_test_amount,
                              scoring = config.grid_scoring)
    
    grid.fit(X_train , y_train)
    return grid

In [ ]:
def custom_model_parameters():
    return {

        "LogisticRegression": {
            "estimator__C": scipy.stats.loguniform(1e-4, 1e2),
            "estimator__penalty": ["l2"],
            "estimator__solver": ["lbfgs"],
        },

        "DecisionTreeClassifier": {
            "estimator__criterion": ["gini", "entropy"],
            "estimator__max_depth": [None, 5, 10, 20],
            "estimator__min_samples_split": scipy.stats.randint(2, 20),
            "estimator__min_samples_leaf": scipy.stats.randint(1, 10),
        },

        "RandomForestClassifier": {
            "estimator__n_estimators": scipy.stats.randint(100, 500),
            "estimator__max_depth": scipy.stats.randint(5, 40),
            "estimator__min_samples_split": scipy.stats.randint(2, 20),
            "estimator__min_samples_leaf": scipy.stats.randint(1, 10),
            "estimator__max_features": ["sqrt", "log2"],
            "estimator__bootstrap": [True, False],
        },

        "LinearSVC": {
            "estimator__C": scipy.stats.loguniform(1e-4, 1e2),
            "estimator__loss": ["hinge", "squared_hinge"],
            "estimator__dual": [True],
            "estimator__max_iter": [5000, 10000],
        },

        "XGBClassifier": {
            "estimator__n_estimators": scipy.stats.randint(100, 500),
            "estimator__learning_rate": scipy.stats.loguniform(1e-3, 0.3),
            "estimator__max_depth": scipy.stats.randint(3, 10),
            "estimator__subsample": scipy.stats.uniform(0.6, 0.4),
            "estimator__colsample_bytree": scipy.stats.uniform(0.6, 0.4),
            "estimator__gamma": scipy.stats.uniform(0, 5),
            "estimator__min_child_weight": scipy.stats.randint(1, 10),
        },

    }


In [ ]:
def data_split():
    X_train , X_test , y_train , y_test = train_test_split(data.x,
                                                           data.y,
                                                           shuffle = True,
                                                           random_state = config.seed,
                                                           test_size = config.test_size,
                                                           stratify = data.y)
    
    return  X_train , X_test , y_train , y_test

In [ ]:
def data_split():
    X_train , X_test , y_train , y_test = train_test_split(data.x,
                                                           data.y,
                                                           shuffle = True,
                                                           random_state = config.seed,
                                                           test_size = config.test_size,
                                                           stratify = data.y)
    
    return  X_train , X_test , y_train , y_test

In [ ]:
def custom_data_validation(estimator , X_train , y_train):
    kfold = StratifiedKFold(n_splits = config.cross_iterations , shuffle = True , random_state = config.seed)

    return cross_validate(estimator = estimator,
                          X= X_train,
                          y= y_train,
                          return_train_score = True,
                          scoring="roc_auc",
                          cv = kfold,
                          return_estimator=True,
                          verbose=config.verbose,
                          )

In [ ]:
def custom_pipeline(estimator , transform = False , smote = False):
    steps = []

    
    if transform:
        steps.append(("TransformPeformance" , DataTransform()))
    steps.append(("BasePeformance" , DataPreprcocess()))

    if smote:
        steps.append(("smote" , SMOTE()))
        
    steps.append(("estimator" , estimator))
    return Pipeline(steps)

In [ ]:
def plot_scores(
        self,
        data,
        x_column,
        score_columns,
        title="Model Comparison",
        ylabel="Score"
    ):
        # Falls eine Liste von Dictionaries übergeben wird
        if isinstance(data, list):
            data = pd.DataFrame(data)

        x = range(len(data))

        plt.figure(figsize=(12, 6))

        for column in score_columns:
            if column in data.columns:
                plt.plot(
                    x,
                    data[column],
                    marker="o",
                    linewidth=2,
                    label=column
                )

        plt.xticks(x, data[x_column], rotation=45)
        plt.xlabel(x_column)
        plt.ylabel(ylabel)
        plt.title(title)
        plt.grid(True, linestyle="--", alpha=0.5)
        plt.legend()
        plt.tight_layout()
        plt.show()
        

In [ ]:
class Benchmark():

    def __init__(self , UseCV = False , UseGrid = False ):
        self.X_train , self.X_test , self.y_train , self.y_test = data_split()
        self.results = []

        self.use_cv = UseCV
        self.use_grid_search = UseGrid
        self.train()

    def train(self):

        for estimator_name , estimators in model_varianz().items():

            base_pipe = custom_pipeline(estimator=estimators , transform= False , smote=False)
            transformed_pipe = custom_pipeline(estimator=estimators , transform= True , smote=True)

            if self.use_cv:
                base_cv = custom_data_validation(estimator=base_pipe,
                                                X_train=self.X_train,
                                                y_train=self.y_train)
                
                transformed_cv = custom_data_validation(estimator=transformed_pipe,
                                                X_train=self.X_train,
                                                y_train=self.y_train)
                print(f"keys;{base_cv.keys()}")
                self.results.append({

                    "base_cv_train_performance":base_cv["train_score"].mean(),
                    "base_cv_train_performance":base_cv["test_score"].mean(),
                    "base_cv_std_performance":base_cv["train_score"].std(),

                    "transformed_cv_train_performance":transformed_cv["train_score"].mean(),
                    "transformed_cv_train_performance":transformed_cv["train_score"].mean(),
                    "transformed_cv_std_performance":transformed_cv["train_score"].std(),
                })

            if self.use_grid_search:
                base_grid = custom_grid_search(estimator=base_pipe,
                                            X_train=self.X_train,
                                            y_train=self.y_train,
                                            parameters=custom_model_parameters()[estimator_name])

                transformed_grid = custom_grid_search(estimator=base_pipe,
                                            X_train=self.X_train,
                                            y_train=self.y_train,
                                            parameters=custom_model_parameters()[estimator_name])
            self.results.append({

                "base_cv_train_performance":base_grid.best_estimator_,
                "transformed_cv_train_performance":transformed_grid.best_estimator_,
            })

        plot_scores(data = self.results)
        
Benchmark(UseGrid=True , UseCV=True)

keys;dict_keys(['fit_time', 'score_time', 'estimator', 'test_score', 'train_score'])


/home/drdrakken/.clean_venv/lib64/python3.14/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/drdrakken/.clean_venv/lib64/python3.14/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/drdrakken/.clean_venv/